In [1]:
# Classificação Automática de Atividades Culturais e Educacionais
# 03 - Preparação do texto com embeddings
#
# Integrantes: Beatriz Aparecida de Mello Barbosa (10354067), Bruna Gonçalves Corte David (10425696),
#              Henrique Brainer Costa (10420717), João Pedro Queiroz de Andrade (10425822),
#              Júlia Andrade (10428513)
#
# Conteúdo: transforma o texto das atividades em embeddings com um modelo de português, anota quanto
#           isso custa de tempo e compara, nos mesmos exemplos, o que o TF-IDF acha parecido e o que
#           o embedding acha parecido.
#
# Alterações:
#   2026-09-25 - Bruna - notebook novo, criado quando voltamos os embeddings para o centro do
#                        projeto, como estava na proposta

O TF-IDF compara duas atividades contando palavras iguais, então duas descrições que falam da mesma
coisa com outras palavras passam batido. A ideia do embedding é diferente: o modelo lê o texto
inteiro e devolve uma lista de números que representa o sentido daquilo. Dois textos sobre o mesmo
assunto ficam com listas parecidas mesmo sem repetir palavra nenhuma. Era essa a aposta que fizemos
na proposta da Parte 1, e é o que testamos aqui.

In [2]:
# ====================================================================================================
#                              CARREGAMENTO DO TEXTO E DO MODELO
# ====================================================================================================
import pandas as pd                                                   # leitura e manipulação das tabelas
import numpy as np                                                    # contas com vetores e matrizes
import time                                                           # mede quanto tempo cada etapa custa
from sentence_transformers import SentenceTransformer                 # carrega modelos que transformam frase em lista de números

activities = pd.read_csv("../data/processed/activities_prepared.csv", sep=";", encoding="utf-8-sig")   # tabela gerada pelo notebook 02

print("Atividades carregadas:", len(activities))

Atividades carregadas: 693


Escolhemos o Serafim 335m, do projeto PORTULAN. Ele é treinado em português, e não num monte de
idiomas ao mesmo tempo, e foi feito para dizer se duas frases querem dizer a mesma coisa, que é
exatamente a conta que precisamos. Também pesou o fato de ele baixar sem cadastro nem senha, porque
quem for rodar este notebook depois precisa conseguir baixar o mesmo modelo. Os outros candidatos
que consideramos, e o motivo de cada descarte, estão na metodologia do artigo.

In [3]:
model_name = "PORTULAN/serafim-335m-portuguese-pt-sentence-encoder"              # modelo de português feito para comparar frases

time_before_loading = time.time()                                                # marca o relógio antes
embedding_model = SentenceTransformer(model_name)                                # baixa e carrega o modelo
loading_time = time.time() - time_before_loading                                 # quanto custou carregar

print("Modelo carregado em", round(loading_time, 1), "segundos")
print("Números em cada vetor:", embedding_model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Modelo carregado em 3.8 segundos
Números em cada vetor: 1024


C:\Users\bruna\AppData\Local\Temp\ipykernel_3960\310115521.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Números em cada vetor:", embedding_model.get_sentence_embedding_dimension())


In [4]:
time_before_encoding = time.time()                                                                      # marca o relógio antes
activity_embeddings = embedding_model.encode(activities["text"].tolist(), show_progress_bar=False)      # um vetor por atividade
encoding_time = time.time() - time_before_encoding                                                      # custo que prometemos medir na Parte 1

print("Atividades transformadas em vetor:", activity_embeddings.shape[0])
print("Tempo para gerar todos:", round(encoding_time, 1), "segundos")

Atividades transformadas em vetor: 693
Tempo para gerar todos: 311.4 segundos


Vale anotar a diferença de custo. O TF-IDF monta a matriz em menos de um segundo, porque só conta
palavra. O modelo de embeddings cobra o tempo acima, mesmo rodando uma vez só. Numa API isso muda o
projeto: o modelo fica carregado na memória e cada atividade nova paga uma fração desse tempo.

In [5]:
# ====================================================================================================
#          O EMBEDDING ENTENDE O QUE O TF-IDF NÃO ENTENDE? COMPARAÇÃO NOS MESMOS EXEMPLOS
# ====================================================================================================
from sklearn.feature_extraction.text import TfidfVectorizer          # a representação por contagem de palavras, para comparar
from sklearn.metrics.pairwise import cosine_similarity               # mede o quanto dois vetores apontam para o mesmo lado
from nltk.corpus import stopwords                                    # palavras comuns do português

portuguese_stopwords = stopwords.words("portuguese")                                      # mesma configuração do notebook 02
tfidf_vectorizer = TfidfVectorizer(stop_words=portuguese_stopwords)                        # TF-IDF igual ao do notebook 02
tfidf_activities = tfidf_vectorizer.fit_transform(activities["text"])                      # mesma base, outra representação

tfidf_similarity = cosine_similarity(tfidf_activities)                                     # quem parece com quem, contando palavras
embedding_similarity = cosine_similarity(activity_embeddings)                              # quem parece com quem, pelo sentido

print("Matrizes de semelhança prontas:", tfidf_similarity.shape)

Matrizes de semelhança prontas: (693, 693)


In [6]:
# ----------------------------------------------------------------------------------------------------
#             AS TRÊS ATIVIDADES MAIS PARECIDAS COM UMA MESMA ATIVIDADE, NOS DOIS MÉTODOS
# ----------------------------------------------------------------------------------------------------
example_activity = 0                                                                                            # a primeira atividade da tabela serve de exemplo

closest_by_tfidf = pd.Series(tfidf_similarity[example_activity]).drop(example_activity).sort_values(ascending=False).head(3)           # as 3 mais parecidas contando palavras
closest_by_embedding = pd.Series(embedding_similarity[example_activity]).drop(example_activity).sort_values(ascending=False).head(3)   # as 3 mais parecidas pelo sentido

print("ATIVIDADE:", activities.loc[example_activity, "title"])
print()
print("Parecidas segundo o TF-IDF:")
for position, value in closest_by_tfidf.items():
    print("   ", round(value, 3), "-", activities.loc[position, "title"])
print()
print("Parecidas segundo o embedding:")
for position, value in closest_by_embedding.items():
    print("   ", round(value, 3), "-", activities.loc[position, "title"])

ATIVIDADE: DESESTRESSAR PINTANDO: A ARTE DE RELAXAR COM LIVROS DE COLORIR – COM EQUIPE BIBLIOTECA

Parecidas segundo o TF-IDF:
    0.157 - Bora trocar livros!
    0.135 - Livros Esquisitos
    0.127 - Livros Brincantes

Parecidas segundo o embedding:
    0.715 - EXPLOSÃO DE CORES – COM EQUIPE BIBLIOTECA
    0.674 - LIVRO-LIVRE: Experiências em gravura e livro de artista
    0.67 - ENTRE CORES E BRINCADEIRAS


In [7]:
# ----------------------------------------------------------------------------------------------------
#                       O MESMO EXAME EM MAIS ALGUMAS ATIVIDADES, PARA NÃO SER SORTE
# ----------------------------------------------------------------------------------------------------
for example_activity in [50, 200, 400, 600]:
    closest_by_tfidf = pd.Series(tfidf_similarity[example_activity]).drop(example_activity).idxmax()            # a mais parecida contando palavras
    closest_by_embedding = pd.Series(embedding_similarity[example_activity]).drop(example_activity).idxmax()    # a mais parecida pelo sentido
    print("=" * 100)
    print("ATIVIDADE:   ", activities.loc[example_activity, "title"])
    print("  TF-IDF:    ", activities.loc[closest_by_tfidf, "title"])
    print("  EMBEDDING: ", activities.loc[closest_by_embedding, "title"])

ATIVIDADE:    OFICINA DE CINE GUERRILHA – COM EQUIPE BIBLIOTECA
  TF-IDF:     ESCREVENDO MINHA CANÇÃO – COM EQUIPE BIBLIOTECA
  EMBEDDING:  Costurando Céus - Criação de vídeos poéticos para redes sociais
ATIVIDADE:    Bê-a-Bá Digital: Módulo II
  TF-IDF:     Projetos Livres
  EMBEDDING:  Introdução à técnicas de gravura
ATIVIDADE:    Técnicas de Didgeridoo
  TF-IDF:     Oficina de construção de Balafon
  EMBEDDING:  Violão Popular - Iniciação
ATIVIDADE:    Design em revolta: cartazes e agitação visual
  TF-IDF:     Serigrafia Artesanal: Do Estêncil à Impressão Manual
  EMBEDDING:  LIVRO-LIVRE: Experiências em gravura e livro de artista


In [8]:
# ----------------------------------------------------------------------------------------------------
#                    OS DOIS MÉTODOS CONCORDAM SOBRE QUEM É PARECIDO COM QUEM?
# ----------------------------------------------------------------------------------------------------
off_diagonal = ~np.eye(len(activities), dtype=bool)                                                                       # ignora a semelhança da atividade com ela mesma
agreement_between_methods = np.corrcoef(tfidf_similarity[off_diagonal], embedding_similarity[off_diagonal])[0, 1]           # o quanto as duas medidas andam juntas

print("Semelhança média pelo TF-IDF:", round(tfidf_similarity[off_diagonal].mean(), 4))
print("Semelhança média pelo embedding:", round(embedding_similarity[off_diagonal].mean(), 4))
print("Correlação entre as duas formas de medir semelhança:", round(agreement_between_methods, 3))

Semelhança média pelo TF-IDF: 0.0196
Semelhança média pelo embedding: 0.3711
Correlação entre as duas formas de medir semelhança: 0.478


Se essa correlação fosse alta, trocar de representação seria detalhe. Sendo baixa, as duas discordam
sobre o que é parecido com o quê, e isso vai aparecer na clusterização como grupos diferentes.

In [9]:
# ====================================================================================================
#                                      GUARDAR OS VETORES
# ====================================================================================================
np.save("../data/processed/embeddings.npy", activity_embeddings)              # vetores usados pela clusterização e pelo classificador

print("Embeddings salvos em data/processed/embeddings.npy")
print("Formato guardado:", activity_embeddings.shape)

Embeddings salvos em data/processed/embeddings.npy
Formato guardado: (693, 1024)
